In [1]:
import os
import warnings
import logging
%pip install -q sktime
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.validation import check_is_fitted
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/mini-rocket-util/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
import importlib

from scipy.stats import randint, uniform, loguniform

tf.keras.backend.clear_session()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 9.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


E0000 00:00:1776020688.443237      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776020688.557076      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776020689.485504      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776020689.485544      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776020689.485546      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776020689.485549      23 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_run_name = 'mini_rocket_v1'
select_k_name = 'select_k_percentile'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
pipe_name = 'temporal_extractor'

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report    = False
save_model   = False
random_search = False
train_size = 0.8

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]
target_only_df = target_only_df[target_only_df['phase'] == 'Gesture']

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=train_size,
    test_pct=0.2
)
# some_sequences = train_sample_df['sequence_id'].unique()[50:]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

Train: 3856 seqs | 75.4%
Test:  629 seqs  | 12.3%


In [5]:
if random_search:
    # ============================================================================
    # RANDOM SEARCH PARAMETER GRID
    # ============================================================================
    param_grid = {
        # ===== RAW SEQUENCE EXTRACTOR =====
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [None],  # TOF harms performance
        f'{pipe_name}__acc_mode':      ['velocity'],
        f'{pipe_name}__rotation_mode': ['quaternion'],
        f'{pipe_name}__thm_mode':      ['delta'],
        f'{pipe_name}__tof_mode':      ['baseline'],
        
        # ===== SEQUENCE BUILDER =====
        'sequence_builder__maxlen':         randint(30, 70),  # Shorter = less overfitting
        'sequence_builder__padding_value':  [-999.0],
        
        # ===== MINIROCKET CORE =====
        'classifier__estimator__num_kernels': randint(300, 1500),  # Fewer = more regularization
        'classifier__estimator__random_state': [42],
        
        # ===== CLASSIFIER TYPE SELECTION =====
        'classifier__estimator__classifier_type': [
            'poisson', 'poisson_nb', 'tweedie',  # Poisson-based models
            'logistic', 'sgd', 'passive',        # Linear classifiers
            'ridge', 'ridge_cv', 'svm',          # Ridge & SVM
            'mlp'                                 # Neural network
        ],
        
        # ===== COMMON PARAMETERS =====
        'classifier__estimator__class_weight': ['balanced'],  # Handle imbalanced data
        'classifier__estimator__max_iter': [1000],
        
        # ===== LOGISTIC REGRESSION PARAMETERS =====
        # C: Inverse regularization (smaller = stronger regularization)
        'classifier__estimator__C': loguniform(0.001, 100),  # For logistic, passive, svm
        'classifier__estimator__penalty': ['l2'],  # L2 regularization
        'classifier__estimator__solver': ['lbfgs', 'saga'],  # lbfgs for l2, saga for elasticnet
        
        # ===== SGD CLASSIFIER PARAMETERS =====
        # alpha: Regularization strength (larger = stronger)
        'classifier__estimator__alpha': loguniform(1e-6, 0.1),  # For sgd, ridge, mlp, poisson, tweedie
        'classifier__estimator__loss': ['log_loss', 'hinge'],  # log_loss for probabilities, hinge for SVM-style
        'classifier__estimator__learning_rate': ['optimal', 'adaptive'],
        
        # ===== PASSIVE AGGRESSIVE PARAMETERS =====
        # C_passive: Regularization (smaller = stronger)
        'classifier__estimator__C_passive': loguniform(0.001, 10),
        
        # ===== RIDGE CLASSIFIER PARAMETERS =====
        # alpha_ridge: Regularization (larger = stronger)
        'classifier__estimator__alpha_ridge': loguniform(0.01, 100),
        
        # ===== RIDGE CV PARAMETERS =====
        'classifier__estimator__alphas': [
            [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
            [0.1, 1.0, 10.0],           # Weak to moderate
            [1.0, 10.0, 50.0, 100.0],   # Moderate to strong (for overfitting)
            [10.0, 50.0, 100.0, 500.0], # Strong regularization
        ],
        
        # ===== SVM PARAMETERS =====
        # C_svm: Inverse regularization (smaller = stronger)
        'classifier__estimator__C_svm': loguniform(0.001, 10),
        'classifier__estimator__loss_svm': ['squared_hinge', 'hinge'],
        
        # ===== MLP PARAMETERS =====
        'classifier__estimator__hidden_layer_sizes': [(50,), (100,), (50, 25), (100, 50)],
        'classifier__estimator__activation': ['relu', 'tanh'],
        'classifier__estimator__alpha_mlp': loguniform(1e-5, 0.01),  # L2 regularization
        'classifier__estimator__learning_rate_init': loguniform(0.0001, 0.01),
        'classifier__estimator__early_stopping': [True],  # Built-in regularization
        
        # ===== POISSON REGRESSION PARAMETERS =====
        # For classifier_type='poisson' only
        'classifier__estimator__alpha_poisson': loguniform(0.01, 10),  # Regularization
        
        # ===== POISSON NAIVE BAYES PARAMETERS =====
        # For classifier_type='poisson_nb' only
        'classifier__estimator__alpha_nb': loguniform(0.1, 10),  # Laplace smoothing
        
        # ===== TWEEDIE REGRESSION PARAMETERS =====
        # For classifier_type='tweedie' only
        'classifier__estimator__power': [1.0, 1.5, 2.0],  # 1=Poisson, 1.5=NegBinom, 2=Gamma
        'classifier__estimator__alpha_tweedie': loguniform(0.01, 10),
    }

else:
    # ============================================================================
    # GRID SEARCH PARAMETER GRID (Full factorial)
    # ============================================================================
    param_grid = {
        # ===== RAW SEQUENCE EXTRACTOR =====
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [None],  # TOF harms performance - keep disabled
        f'{pipe_name}__acc_mode':      ['velocity'],
        f'{pipe_name}__rotation_mode': ['quaternion'],
        f'{pipe_name}__thm_mode':      ['delta'],
        f'{pipe_name}__tof_mode':      ['baseline'],
        
        # ===== SEQUENCE BUILDER =====
        'sequence_builder__maxlen':         [55],  # Test different sequence lengths
        'sequence_builder__padding_value':  [-999.0],
        
        # ===== MINIROCKET CORE =====
        'classifier__estimator__num_kernels': [1200],  # Fewer = more regularization
        'classifier__estimator__random_state': [42],
        
        # ===== CLASSIFIER TYPE SELECTION =====
        'classifier__estimator__classifier_type': [
            # 'logistic',  # Start with logistic regression (best balance)
            # 'poisson',  # Uncomment for count data
            # 'poisson_nb',  # Uncomment for classification with count features
            # 'tweedie',  # Uncomment for overdispersed count data
            # 'sgd',  # Uncomment for large datasets
            'ridge',  # Uncomment for fast training
            # 'svm',  # Uncomment for max-margin classification
            # 'mlp',  # Uncomment for non-linear patterns (slow)
        ],
        
        # # ===== COMMON PARAMETERS =====
        # 'classifier__estimator__class_weight': ['balanced'],  # Handle imbalanced classes
        # 'classifier__estimator__
        # 
        # 
        # 
        # ': [1000],
        
        # # ===== LOGISTIC REGRESSION (classifier_type='logistic') =====
        # # C: Inverse regularization (0.01=strong, 0.1=moderate, 1=weak, 10=minimal)
        # 'classifier__estimator__C': [0.01, 0.1, 1.0, 10.0],
        # 'classifier__estimator__penalty': ['l2'],  # L2 regularization only
        # 'classifier__estimator__solver': ['lbfgs'],  # Good for L2 penalty
        
        # # ===== SGD CLASSIFIER (classifier_type='sgd') =====
        # # alpha: Regularization strength (0.0001=weak, 0.001=moderate, 0.01=strong)
        # 'classifier__estimator__alpha': [0.0001, 0.001, 0.01],
        # 'classifier__estimator__loss': ['log_loss', 'hinge'],  # log_loss for probabilities
        # 'classifier__estimator__penalty_sgd': ['l2'],  # L2 regularization
        
        # # ===== PASSIVE AGGRESSIVE (classifier_type='passive') =====
        # # C_passive: Regularization (0.01=strong, 0.1=moderate, 1=weak)
        # 'classifier__estimator__C_passive': [0.01, 0.1, 1.0],
        
        # ===== RIDGE CLASSIFIER (classifier_type='ridge') =====
        # alpha_ridge: Regularization (0.1=weak, 1=moderate, 10=strong, 100=very strong)
        'classifier__estimator__alpha_ridge': [300.0, 500.0, 1000.0],
        
        # # ===== SVM (classifier_type='svm') =====
        # # C_svm: Inverse regularization (0.01=strong, 0.1=moderate, 1=weak)
        # 'classifier__estimator__C_svm': [0.01, 0.1, 1.0],
        # 'classifier__estimator__loss_svm': ['squared_hinge'],  # Faster than hinge
        
        # # ===== MLP CLASSIFIER (classifier_type='mlp') =====
        # # hidden_layer_sizes: Architecture (smaller = less overfitting)
        # 'classifier__estimator__hidden_layer_sizes': [(50,), (100,), (50, 25)],
        # 'classifier__estimator__activation': ['relu'],  # Standard choice
        # 'classifier__estimator__alpha_mlp': [0.0001, 0.001, 0.01],  # L2 regularization
        # 'classifier__estimator__learning_rate_init': [0.0001, 0.001],
        # 'classifier__estimator__early_stopping': [True],  # Prevents overfitting
        
        # # ===== POISSON REGRESSION (classifier_type='poisson') =====
        # # For count data where variance = mean
        # 'classifier__estimator__alpha_poisson': [0.01, 0.1, 1.0, 10.0],
        
        # # ===== POISSON NAIVE BAYES (classifier_type='poisson_nb') =====
        # # For classification with count features (e.g., word counts, sensor readings)
        # 'classifier__estimator__alpha_nb': [0.1, 0.5, 1.0, 2.0],  # Laplace smoothing
        
        # # ===== TWEEDIE REGRESSION (classifier_type='tweedie') =====
        # # For overdispersed count data (variance > mean)
        # 'classifier__estimator__power': [1.0, 1.5, 2.0],  # 1=Poisson, 1.5=NegBinom, 2=Gamma
        # 'classifier__estimator__alpha_tweedie': [0.01, 0.1, 1.0, 10.0],
    }

In [6]:
importlib.reload(utils)

raw_extractor = utils.RawSequenceExtractor(
    acc_cols=acc_columns,
)

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), make_column_selector(pattern='acc|rot|thm|tof')),
], remainder='drop', verbose_feature_names_out=False)
preprocessor.set_output(transform='pandas')

sequence_builder = utils.SequencePadder(maxlen=110, padding_value=-999.0)

# IMPORTANT: Use create_rocket_classifier with default params
# The grid search will override them
rocket_clf = utils.create_rocket_classifier(
    classifier_type='ridge',  # Default, will be overridden by grid search
    num_kernels=1000,
    random_state=42
)

classifier = utils.ManyToOneWrapperRNN(estimator=rocket_clf, target='gesture_action')

pipeline = Pipeline([
    (pipe_name, raw_extractor),
    ('preprocessor', preprocessor),
    ('sequence_builder', sequence_builder),
    ('classifier', classifier),
])

cv = GroupKFold(n_splits=n_splits)

In [7]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=7,
                cv=cv,
                random_state=42,
                n_jobs=1,
                verbose=3,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=3,
                n_jobs=1,
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']

        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)

    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    master_cv_results_df['train_size'] = train_size
    master_cv_results_df['is_random_search'] = random_search
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

Fitting 3 folds for each of 3 candidates, totalling 9 fits
[CV 1/3] END classifier__estimator__alpha_ridge=300.0, classifier__estimator__classifier_type=ridge, classifier__estimator__num_kernels=1200, classifier__estimator__random_state=42, sequence_builder__maxlen=55, sequence_builder__padding_value=-999.0, temporal_extractor__acc_cols=['acc_x', 'acc_y', 'acc_z'], temporal_extractor__acc_mode=velocity, temporal_extractor__rot_cols=['rot_w', 'rot_x', 'rot_y', 'rot_z'], temporal_extractor__rotation_mode=quaternion, temporal_extractor__thm_cols=['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5'], temporal_extractor__thm_mode=delta, temporal_extractor__tof_cols=None, temporal_extractor__tof_mode=baseline;, score=(train=1.000, test=0.698) total time= 1.2min
[CV 2/3] END classifier__estimator__alpha_ridge=300.0, classifier__estimator__classifier_type=ridge, classifier__estimator__num_kernels=1200, classifier__estimator__random_state=42, sequence_builder__maxlen=55, sequence_builder__padding_value